In [ ]:
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import kpss, InterpolationWarning
from sklearn.metrics import mean_squared_error, r2_score
import os
import matplotlib.pyplot as plt
import warnings
import numpy as np

# Especificar o caminho para a pasta onde os arquivos estão localizados
caminho_pasta = '/Users/arielportela/Documents/Workspace/mestrado/Vazão Traceroutes Recorrentes/Não Recorrentes'

# Função para carregar os datasets e identificar pares de arquivos BBR e Cubic
def carregar_datasets(caminho):
    arquivos = os.listdir(caminho)
    datasets_bbr = {}
    datasets_cubic = {}
    
    for arquivo in arquivos:
        if arquivo.endswith('.csv'):
            nome = arquivo.split('.')[0].strip()  # Remover espaços extras
            df = pd.read_csv(os.path.join(caminho, arquivo))
            if 'bbr' in nome:
                datasets_bbr[nome] = df
            elif 'cubic' in nome:
                datasets_cubic[nome] = df
                
    return datasets_bbr, datasets_cubic

# Carregar todos os datasets
datasets_bbr, datasets_cubic = carregar_datasets(caminho_pasta)

# Imprimir nomes carregados para verificação
print(f"Datasets BBR carregados: {list(datasets_bbr.keys())}")
print(f"Datasets Cubic carregados: {list(datasets_cubic.keys())}")

# Função para decompor a série em tendência e sazonalidade
def decompor_serie(df, coluna='Vazao', periodo=4):
    df = df.dropna(subset=[coluna])  # Remover NaNs
    if len(df) >= periodo:
        result = seasonal_decompose(df[coluna], model='additive', period=periodo)
        return result.trend, result.seasonal
    else:
        print(f"Dados insuficientes para decomposição na coluna {coluna}")
        return None, None

# Função para calcular correlação de tendência e sazonalidade entre datasets "contrários"
def calcular_correlacao_sazonal_tendencia(df1, df2, coluna='Vazao', periodo=4):
    tendencia1, sazonalidade1 = decompor_serie(df1, coluna, periodo)
    tendencia2, sazonalidade2 = decompor_serie(df2, coluna, periodo)
    if tendencia1 is not None and tendencia2 is not None:
        # Alinhar séries temporais para garantir o mesmo número de amostras
        tendencia1, tendencia2 = tendencia1.align(tendencia2, join='inner')
        sazonalidade1, sazonalidade2 = sazonalidade1.align(sazonalidade2, join='inner')
        
        # Remover NaNs após o alinhamento
        tendencia1 = tendencia1.dropna()
        tendencia2 = tendencia2.dropna()
        sazonalidade1 = sazonalidade1.dropna()
        sazonalidade2 = sazonalidade2.dropna()
        
        # Garantir que os índices estejam alinhados após a remoção dos NaNs
        tendencia1, tendencia2 = tendencia1.align(tendencia2, join='inner')
        sazonalidade1, sazonalidade2 = sazonalidade1.align(sazonalidade2, join='inner')
        
        # Calcular correlações
        correlacao_tendencia = tendencia1.corr(tendencia2)
        correlacao_sazonalidade = sazonalidade1.corr(sazonalidade2)
        
        # Cálculo do RMSE
        rmse_tendencia = np.sqrt(mean_squared_error(tendencia1, tendencia2))
        rmse_sazonalidade = np.sqrt(mean_squared_error(sazonalidade1, sazonalidade2))
        
        # Cálculo da "acurácia" (usando o R² como proxy)
        r2_tendencia = r2_score(tendencia1, tendencia2)
        r2_sazonalidade = r2_score(sazonalidade1, sazonalidade2)
        
        return (correlacao_tendencia, correlacao_sazonalidade, tendencia1, tendencia2,
                sazonalidade1, sazonalidade2, rmse_tendencia, rmse_sazonalidade, r2_tendencia, r2_sazonalidade)
    else:
        return None, None, None, None, None, None, None, None, None, None

# Dicionário para armazenar correlações
correlacoes = {}

# Iterar sobre os datasets para calcular correlações entre pares BBR e Cubic
for nome_bbr in datasets_bbr.keys():
    nome_cubic = nome_bbr.replace('bbr', 'cubic').strip()  # Remover espaços extras
    if nome_cubic in datasets_cubic:
        df_bbr = datasets_bbr[nome_bbr]
        df_cubic = datasets_cubic[nome_cubic]
        localidade = nome_bbr.replace('vazao bbr data', '').replace(' 2023', '').upper().strip()
        correlacao_resultado = calcular_correlacao_sazonal_tendencia(df_cubic, df_bbr, periodo=4)
        if correlacao_resultado[0] is not None:
            correlacoes[localidade] = correlacao_resultado
        else:
            print(f"Não foi possível calcular correlações para {nome_bbr}")
    else:
        print(f"Chave não encontrada para o par: {nome_cubic}")

# Função para realizar o teste KPSS para cada dataset
def teste_kpss(df, coluna='Vazao', diferenciar=False):
    df = df[[coluna]].dropna()
    if len(df) > 0:  # Verificar se há dados suficientes para o teste
        if diferenciar:
            df = df.diff().dropna()  # Diferenciação para remover tendência
        try:
            with warnings.catch_warnings(record=True) as w:
                warnings.simplefilter('always')
                kpss_stat, p_value, _, _ = kpss(df[coluna], regression='c')
                if any(issubclass(warn.category, InterpolationWarning) for warn in w):
                    print(f"InterpolationWarning para {coluna} - P-valor ajustado")
                    p_value = 0.01 if kpss_stat > 0.739 else 0.1
                conclusao = 'Não estacionária' if p_value < 0.05 else 'Estacionária'
                return kpss_stat, p_value, conclusao
        except ValueError as e:
            print(f"Erro ao realizar o teste KPSS: {e}")
            return None, None, 'Erro'
    else:
        print(f"Dados insuficientes para o teste KPSS na coluna {coluna}")
        return None, None, 'Dados insuficientes'

# Realizar o teste KPSS para cada dataset com diferenciação para remover tendência, se necessário
resultados_kpss = {}
for nome, df in {**datasets_bbr, **datasets_cubic}.items():
    resultados_kpss[nome] = teste_kpss(df, diferenciar=True)

# Verificar quais chaves existem em resultados_kpss
print(f"Chaves de resultados_kpss: {list(resultados_kpss.keys())}")

# Montar uma tabela com todos os resultados
resultados = {
    'Dataset': [],
    'Correlação da Tendência': [],
    'Correlação Sazonal': [],
    #'RMSE Tendência': [],
    #'RMSE Sazonal': [],
    #'Acurácia Tendência (R²)': [],
    #'Acurácia Sazonalidade (R²)': [],
    'KPSS Estatística': [],
    'p-valor': [],
    'Conclusão': []
}

for localidade, correlacao in correlacoes.items():
    cubic_nome = f'vazao cubic data {localidade.lower()} 2023'.strip()
    bbr_nome = f'vazao bbr data {localidade.lower()} 2023'.strip()
    
    # Verificar se as chaves existem antes de acessar os resultados_kpss
    if cubic_nome in resultados_kpss and bbr_nome in resultados_kpss:
        # Adicionando resultados para Cubic vs BBR
        resultados['Dataset'].append(f'{localidade} - Cubic')
        resultados['Correlação da Tendência'].append(correlacao[0])
        resultados['Correlação Sazonal'].append(correlacao[1])
        #resultados['RMSE Tendência'].append(correlacao[6])
        #resultados['RMSE Sazonal'].append(correlacao[7])
        #resultados['Acurácia Tendência (R²)'].append(correlacao[8])
        #resultados['Acurácia Sazonalidade (R²)'].append(correlacao[9])
        resultados['KPSS Estatística'].append(resultados_kpss[cubic_nome][0])
        resultados['p-valor'].append(resultados_kpss[cubic_nome][1])
        resultados['Conclusão'].append(f'Cubic - {resultados_kpss[cubic_nome][2]}')
        
        # Adicionando resultados para BBR vs Cubic
        resultados['Dataset'].append(f'{localidade} - BBR')
        resultados['Correlação da Tendência'].append(correlacao[0])
        resultados['Correlação Sazonal'].append(correlacao[1])
        #resultados['RMSE Tendência'].append(correlacao[6])
        #resultados['RMSE Sazonal'].append(correlacao[7])
        #resultados['Acurácia Tendência (R²)'].append(correlacao[8])
        #resultados['Acurácia Sazonalidade (R²)'].append(correlacao[9])
        resultados['KPSS Estatística'].append(resultados_kpss[bbr_nome][0])
        resultados['p-valor'].append(resultados_kpss[bbr_nome][1])
        resultados['Conclusão'].append(f'BBR - {resultados_kpss[bbr_nome][2]}')
    else:
        print(f"Chaves não encontradas: {cubic_nome} ou {bbr_nome}")

# Criar o DataFrame e remover as linhas onde Correlação da Tendência ou Sazonalidade é NaN
resultados_df = pd.DataFrame(resultados)
resultados_df = resultados_df.dropna(subset=['Correlação da Tendência', 'Correlação Sazonal'], how='all')

print(resultados_df)

# Gerar gráficos para a correlação de tendência e sazonalidade
for localidade, correlacao in correlacoes.items():
    tendencia1 = correlacao[2]
    tendencia2 = correlacao[3]
    sazonalidade1 = correlacao[4]
    sazonalidade2 = correlacao[5]

    # if tendencia1 is not None and tendencia2 is not None:
    #     # Gráfico de correlação de tendência
    #     plt.figure(figsize=(14, 5))
    #     plt.subplot(1, 2, 1)
    #     plt.plot(tendencia1, label='Cubic Tendência', linestyle='--')
    #     plt.plot(tendencia2, label='BBR Tendência', linestyle='--')
    #     plt.title(f'{localidade} - Correlação de Tendência Recorrente')
    #     plt.legend()
        
    #     # Gráfico de correlação sazonal
    #     plt.subplot(1, 2, 2)
    #     plt.plot(sazonalidade1, label='Cubic Sazonalidade')
    #     plt.plot(sazonalidade2, label='BBR Sazonalidade')
    #     plt.title(f'{localidade} - Correlação Sazonal')
    #     plt.legend()
        
    #     plt.tight_layout()
    #     plt.show()


In [ ]:
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import kpss, InterpolationWarning
from sklearn.metrics import mean_squared_error, r2_score
import os
import matplotlib.pyplot as plt
import warnings
import numpy as np

In [ ]:
# Decomposing time series into trend and seasonality
def series_decomposition(df, column='Throughput', period=4):
    df = df.dropna(subset=[column])  # Remove NaNs
    if len(df) >= period:
        result = seasonal_decompose(df[column], model='additive', period=period)
        return result.trend, result.seasonal
    else:
        print(f"Insufficient data for column {column}")
        return None, None

# Calculating trend and seasonality correlation between datasets
def calculate_seasonal_trend_correlation(df1, df2, column='Throughput', periodo=4):
    trend1, seasonality1 = series_decomposition(df1, column, periodo)
    trend2, seasonality2 = series_decomposition(df2, column, periodo)
    if trend1 is not None and trend2 is not None:
        # Align series to same size
        trend1, trend2 = trend1.align(trend2, join='inner')
        seasonality1, seasonality2 = seasonality1.align(seasonality2, join='inner')
        
        # Remove NaNs
        trend1 = trend1.dropna()
        trend2 = trend2.dropna()
        seasonality1 = seasonality1.dropna()
        seasonality2 = seasonality2.dropna()
        
        # Align indexes
        trend1, trend2 = trend1.align(trend2, join='inner')
        seasonality1, seasonality2 = seasonality1.align(seasonality2, join='inner')
        
        # Corralation calculus
        correlacao_tendencia = trend1.corr(trend2)
        correlacao_sazonalidade = seasonality1.corr(seasonality2)
        
        # RMSE calculus
        rmse_tendencia = np.sqrt(mean_squared_error(trend1, trend2))
        rmse_sazonalidade = np.sqrt(mean_squared_error(seasonality1, seasonality2))
        
        # Accuracy calculus (using R² as proxy)
        r2_tendencia = r2_score(trend1, trend2)
        r2_sazonalidade = r2_score(seasonality1, seasonality2)
        
        return (correlacao_tendencia, correlacao_sazonalidade, trend1, trend2,
                seasonality1, seasonality2, rmse_tendencia, rmse_sazonalidade, r2_tendencia, r2_sazonalidade)
    else:
        return None, None, None, None, None, None, None, None, None, None
    
# Load datasets that are going to be compared (original and imputed) for each imputation_method
def load_datasets(imputed_folder_path, original_folder_path):
    files = os.listdir(imputed_folder_path)
    imputed_datasets = {}
    original_datasets = {}
    
    for file in files:
        if file.endswith('.csv'):
            name = file # Write the name that identifies the link

            df_imputed = pd.read_csv(os.path.join(imputed_folder_path, file))
            imputed_datasets[name] = df_imputed # Save imputed df on list

            df_original = pd.read_csv(os.path.join(original_folder_path, file))
            original_datasets[name] = df_original # Save original df (with missing values) on list 
     
    return imputed_datasets, original_datasets

In [ ]:
imputed_folder = '../datasets/imputed-choosen-best-svd' # Folder where all the dataset imputed are
original_folder = '../datasets/choosen-best-svd' # Folder where all the datasets with missing values are
imputation_methods = ['interpolacao-linear', 'knn', 'media-movel', 'mediana-movel', 'svd'] # Imputation techiniques that are in the imputed folder

In [ ]:


imputed_datasets, original_datasets = load_datasets(imputed_folder, original_folder)

results_kpss = {}
for name, df in {**imputed_datasets, **original_datasets}.items():
    results_kpss[name] = teste_kpss(df, diferenciar=True)

# Verify wich keys are on results_kpss
# print(f"Kpss results keys: {list(results_kpss.keys())}")

correlations = {}

for name in imputed_datasets.keys():
    if name in original_datasets:
        df_imputed = imputed_datasets[name]
        df_original = original_datasets[name]
        result_correlation = calculate_seasonal_trend_correlation(df_imputed, df_original, periodo=4)
        if result_correlation[0] is not None:
            correlacoes[name] = result_correlation
        else:
            print(f"Não foi possível calcular correlações para {name}")
    else:
        print(f"Chave não encontrada para o par: {name}")

# Função para realizar o teste KPSS para cada dataset
def teste_kpss(df, coluna='Vazao', diferenciar=False):
    df = df[[coluna]].dropna()
    if len(df) > 0:  # Verificar se há dados suficientes para o teste
        if diferenciar:
            df = df.diff().dropna()  # Diferenciação para remover tendência
        try:
            with warnings.catch_warnings(record=True) as w:
                warnings.simplefilter('always')
                kpss_stat, p_value, _, _ = kpss(df[coluna], regression='c')
                if any(issubclass(warn.category, InterpolationWarning) for warn in w):
                    print(f"InterpolationWarning para {coluna} - P-valor ajustado")
                    p_value = 0.01 if kpss_stat > 0.739 else 0.1
                conclusao = 'Não estacionária' if p_value < 0.05 else 'Estacionária'
                return kpss_stat, p_value, conclusao
        except ValueError as e:
            print(f"Erro ao realizar o teste KPSS: {e}")
            return None, None, 'Erro'
    else:
        print(f"Dados insuficientes para o teste KPSS na coluna {coluna}")
        return None, None, 'Dados insuficientes'

results = {
    'Dataset': [],
    'Trend correlation': [],
    'Seazonal correlation': [],
    #'RMSE Trend': [],
    #'Seasonal RMSE': [],
    #'Trend accuracy (R²)': [],
    #'Seasonality accuracy (R²)': [],
    'KPSS': [],
    'p-value': [],
    'Conclusion': []
}

for name in os.listdir(original_folder):
    # cubic_nome = f'vazao cubic data {localidade.lower()} 2023'.strip()
    # bbr_nome = f'vazao bbr data {localidade.lower()} 2023'.strip()
    
    # Verificar se as chaves existem antes de acessar os resultados_kpss
    if nome in resultados_kpss:
        # Adicionando resultados para Cubic vs BBR
        resultados['Dataset'].append(f'{localidade} - Cubic')
        resultados['Correlação da Tendência'].append(correlacao[0])
        resultados['Correlação Sazonal'].append(correlacao[1])
        #resultados['RMSE Tendência'].append(correlacao[6])
        #resultados['RMSE Sazonal'].append(correlacao[7])
        #resultados['Acurácia Tendência (R²)'].append(correlacao[8])
        #resultados['Acurácia Sazonalidade (R²)'].append(correlacao[9])
        resultados['KPSS Estatística'].append(resultados_kpss[cubic_nome][0])
        resultados['p-valor'].append(resultados_kpss[cubic_nome][1])
        resultados['Conclusão'].append(f'Cubic - {resultados_kpss[cubic_nome][2]}')
        
        # Adicionando resultados para BBR vs Cubic
        resultados['Dataset'].append(f'{localidade} - BBR')
        resultados['Correlação da Tendência'].append(correlacao[0])
        resultados['Correlação Sazonal'].append(correlacao[1])
        #resultados['RMSE Tendência'].append(correlacao[6])
        #resultados['RMSE Sazonal'].append(correlacao[7])
        #resultados['Acurácia Tendência (R²)'].append(correlacao[8])
        #resultados['Acurácia Sazonalidade (R²)'].append(correlacao[9])
        resultados['KPSS Estatística'].append(resultados_kpss[bbr_nome][0])
        resultados['p-valor'].append(resultados_kpss[bbr_nome][1])
        resultados['Conclusão'].append(f'BBR - {resultados_kpss[bbr_nome][2]}')
    else:
        print(f"Chaves não encontradas: {cubic_nome} ou {bbr_nome}")

# Criar o DataFrame e remover as linhas onde Correlação da Tendência ou Sazonalidade é NaN
resultados_df = pd.DataFrame(resultados)
resultados_df = resultados_df.dropna(subset=['Correlação da Tendência', 'Correlação Sazonal'], how='all')

print(resultados_df)

# Gerar gráficos para a correlação de tendência e sazonalidade
for localidade, correlacao in correlacoes.items():
    tendencia1 = correlacao[2]
    tendencia2 = correlacao[3]
    sazonalidade1 = correlacao[4]
    sazonalidade2 = correlacao[5]

    # if tendencia1 is not None and tendencia2 is not None:
    #     # Gráfico de correlação de tendência
    #     plt.figure(figsize=(14, 5))
    #     plt.subplot(1, 2, 1)
    #     plt.plot(tendencia1, label='Cubic Tendência', linestyle='--')
    #     plt.plot(tendencia2, label='BBR Tendência', linestyle='--')
    #     plt.title(f'{localidade} - Correlação de Tendência Recorrente')
    #     plt.legend()
        
    #     # Gráfico de correlação sazonal
    #     plt.subplot(1, 2, 2)
    #     plt.plot(sazonalidade1, label='Cubic Sazonalidade')
    #     plt.plot(sazonalidade2, label='BBR Sazonalidade')
    #     plt.title(f'{localidade} - Correlação Sazonal')
    #     plt.legend()
        
    #     plt.tight_layout()
    #     plt.show()